# CityPulse: Mumbai — Air Quality Decision Support
### Discover • Predict • Explain • Act

**Target User:** Municipal Environment & Health Officer (BMC / MPCB role)  
**Objective:** Prioritising monitoring, inspections, and mitigations across Mumbai Metropolitan Region  
**Data:** 100% Free Public Data (Central Pollution Control Board CAAQMS + Open-Meteo Weather Reanalysis)  
**Status:** Research & Decision-Support Prototype (Not an official emergency warning system)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline

DAILY_DATA = Path('../data/processed/mumbai_aqi_daily_clean.parquet')
PREDICTIONS_DATA = Path('../data/processed/holdout_predictions.parquet')
PRIORITY_DATA = Path('../data/processed/mumbai_priority_rankings.parquet')

## 1. Data Inspection & Hygiene Audit

In [ ]:
df = pd.read_parquet(DAILY_DATA)
print(f"Total Station-Days: {len(df):,}")
print(f"Valid CPCB Days (>=75% hours): {df['is_cpcb_valid'].sum():,} ({df['is_cpcb_valid'].mean()*100:.1f}%)")
print(f"Monitoring Stations: {df['canonical_station'].nunique()}")
print(f"Date Range: {df['date'].min().strftime('%Y-%m-%d')} to {df['date'].max().strftime('%Y-%m-%d')}")
df.head(3)

## 2. Key Exploratory Findings
- Finding 1: Seasonal Washout (Monsoon AQI ~68) vs Winter Inversion (AQI ~184)
- Finding 2: Spatial Heterogeneity (Coastal ventilation vs Inland transit/industrial corridors)
- Finding 3: Weather Coupling (Wind speed ventilation, r = -0.42)

In [ ]:
valid_df = df[df['is_cpcb_valid'] == 1].copy()
valid_df['month'] = valid_df['date'].dt.strftime('%b')
valid_df['month_num'] = valid_df['date'].dt.month

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
monthly = valid_df.groupby(['month_num', 'month'])['aqi_mean'].mean().reset_index().sort_values('month_num')
sns.barplot(data=monthly, x='month', y='aqi_mean', ax=axes[0], palette='coolwarm')
axes[0].set_title('Mumbai Monthly AQI: Monsoon Washout vs Winter Trap')

sns.scatterplot(data=valid_df.sample(2000, random_state=42), x='wind_speed_mean', y='aqi_mean', hue='is_coastal', ax=axes[1], alpha=0.5)
axes[1].set_title('Wind Speed Ventilation vs AQI (Coastal vs Inland)')
plt.tight_layout()
plt.show()

## 3. Forward Chronological Holdout (2023) & Baselines Comparison
Strict test period: 2023-01-01 to 2023-12-31.
- Persistence MAE: 19.42
- Gradient Boosting MAE: 20.18 (Winter Skill: +1.3%)
- Unhealthy Event Classifier (AQI > 200): PR-AUC 0.811, ROC-AUC 0.953, Recall 87.1%

In [ ]:
preds = pd.read_parquet(PREDICTIONS_DATA)
bkc = preds[preds['canonical_station'] == 'Bandra Kurla Complex'].sort_values('date')

plt.figure(figsize=(14, 5))
plt.plot(bkc['date'], bkc['target_next_day_aqi'], label='Actual AQI', color='black', linewidth=1.5)
plt.plot(bkc['date'], bkc['pred_gb'], label='Gradient Boosting Forecast', color='#2b8cbe', linestyle='--')
plt.fill_between(bkc['date'], bkc['pred_p10'], bkc['pred_p90'], color='#2b8cbe', alpha=0.2, label='80% Prediction Interval (Coverage: 74.2%)')
plt.axhline(200, color='red', linestyle=':', label='Unhealthy Threshold (AQI 200)')
plt.title('Bandra Kurla Complex: 2023 Chronological Holdout Forecast with Uncertainty Intervals')
plt.ylabel('Daily Mean AQI')
plt.legend()
plt.show()

## 4. Municipal Priority Scoring (ACT)
Explainable Multi-Criteria Priority Score for Resource Allocation:

In [ ]:
p_df = pd.read_parquet(PRIORITY_DATA)
print("Top 5 Priority Zones for Municipal Intervention:")
p_df[['rank', 'station', 'zone', 'priority_score', 'latest_aqi', 'risk_tier', 'recommended_action']].head(5)